# исследование: шаги 2–9

Шаг 2. Загрузка и разбиение на обучение и тест 

In [52]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split


df = pd.read_csv("../data/astana_rent_class.csv") 
print(df.shape) 
train, test = train_test_split(df, test_size=0.25, random_state=42) 
print(train.shape, test.shape) 


(600, 11)
(450, 11) (150, 11)


Шаг 3. Парная регрессия по формуле 

In [53]:
from sklearn.linear_model import LinearRegression 
  
tr = train.dropna(subset=["area"]) 
X = np.column_stack([np.ones(len(tr)), tr["area"]]) 
y = tr["price"].to_numpy() 

w = np.linalg.solve(X.T @ X, X.T @ y)      # (XᵀX) w = Xᵀy 
print(w.round(1)) 

lr = LinearRegression().fit(tr[["area"]], y) 
print(round(lr.intercept_, 1), lr.coef_.round(1))

[15056.3  3691. ]
15056.3 [3691.]


Шаг 4. MSE в векторной форме и качество на тесте 

In [54]:
from sklearn.metrics import mean_squared_error, r2_score 

mse = np.square(X @ w - y).mean() 

print(round(mse), round(np.sqrt(mse))) 

te = test.dropna(subset=["area"]) 

pred = lr.predict(te[["area"]]) 

print("test RMSE", round(np.sqrt(mean_squared_error(te["price"], pred))), "R2", round(r2_score(te["price"], pred), 3)) 


1894803898 43529
test RMSE 45171 R2 0.768


Шаг 5. Множественная регрессия в Pipeline 

In [55]:
from sklearn.model_selection import cross_val_score 
from sklearn.pipeline import make_pipeline 
from sklearn.compose import make_column_transformer 
from sklearn.impute import SimpleImputer 
from sklearn.preprocessing import StandardScaler, OneHotEncoder 
  
num_cols = ["rooms", "area", "floor", "floors", "year", "dist_center"] 
cat_cols = ["district", "furniture"] 
X_train, y_train = train[num_cols + cat_cols], train["price"] 
X_test,  y_test  = test[num_cols + cat_cols],  test["price"] 
  
def make_prep(num=num_cols): 
    return make_column_transformer( 
        (make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num), 
        (make_pipeline(SimpleImputer(strategy="most_frequent"), 
                       OneHotEncoder(handle_unknown="ignore")), cat_cols)) 
  
model = make_pipeline(make_prep(), LinearRegression()) 
scores = cross_val_score(model, X_train, y_train, cv=5, scoring="r2") 
print(scores.round(3), scores.mean().round(3)) 
model.fit(X_train, y_train) 
print("test R2", round(model.score(X_test, y_test), 3)) 
  
names = [n.split("__")[1] for n in model[:-1].get_feature_names_out()] 
coefs = pd.Series(model[-1].coef_, index=names) 
print(coefs.round(0).sort_values())

[0.865 0.821 0.878 0.851 0.703] 0.824
test R2 0.871
district_Алматы     -21733.0
district_Байконур   -19289.0
district_Сарыарка   -13709.0
dist_center         -10705.0
furniture_no          -374.0
furniture_yes          374.0
floor                  687.0
year                  1685.0
floors                1759.0
rooms                13014.0
district_Нура        13463.0
district_Есиль       41268.0
area                 57252.0
dtype: float64


Шаг 6. Нелинейный признак: бинаризация расстояния 

In [56]:
from sklearn.preprocessing import KBinsDiscretizer 

num_wo = ["rooms", "area", "floor", "floors", "year"] 

def make_prep_bins(n_bins=8): 
    return make_column_transformer( 
        (make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_wo), 
        (KBinsDiscretizer(n_bins=n_bins, encode="onehot", strategy="uniform"), 
        ["dist_center"]), 
        (make_pipeline(SimpleImputer(strategy="most_frequent"), 
        OneHotEncoder(handle_unknown="ignore")), cat_cols)) 

model_b = make_pipeline(make_prep_bins(), LinearRegression()) 

scores = cross_val_score(model_b, X_train, y_train, cv=5, scoring="r2") 

print(scores.round(3), scores.mean().round(3)) 

model_b.fit(X_train, y_train) 

print("test", round(model_b.score(X_test, y_test), 3)) 

[0.862 0.816 0.873 0.84  0.7  ] 0.818
test 0.873
